# Learned Example

In [39]:

class MyProperty:
    def __init__(self, fget=None, fset=None, fdel=None):
        self.fget = fget
        self.fset = fset
        self.fdel = fdel

    def __get__(self, instance, owner):
        if instance is None:
            return self
        if self.fget is None:
            raise AttributeError("unreadable attribute")
        return self.fget(instance)

    def __set__(self, instance, value):
        if self.fset is None:
            raise AttributeError("can't set attribute")
        self.fset(instance, value)

    def __delete__(self, instance):
        if self.fdel is None:
            raise AttributeError("can't delete attribute")
        self.fdel(instance)

    def setter(self, fset):

        return MyProperty(self.fget, fset, self.fdel)

    def deleter(self, fdel):
        return MyProperty(self.fget, self.fset, fdel)


# ---------- Circle using the custom property ----------
class Circle:
    def __init__(self, radius):
        self.radius = radius          # goes through the setter (validation!)

    @MyProperty
    def radius(self):
        return self._radius

    @radius.setter
    def radius(self, value):
        if value <= 0:
            raise ValueError("Radius must be positive")
        self._radius = value

    @MyProperty
    def area(self):
        return 3.14159 * self._radius ** 2

    @MyProperty
    def circumference(self):
        return 2 * 3.14159 * self._radius


# ---------- Try it ----------
c = Circle(2)
print(c.radius)           # 5
print(c.area)             # 78.53975
print(c.circumference)    # 31.4159

c.radius = 10
print(c.radius)           # 10
print(c.area)             # 314.159
print(c.circumference)    # 62.8318

try:
    c.radius = -1
except ValueError as e:
    print("ValueError:", e)   # Radius must be positive

try:
    c.area = 999
except AttributeError as e:
    print("AttributeError:", e)   # can't set attribute

print(Circle.area)
   # <MyProperty object ...>  ← __get__ with instance=None
print(Circle(4).area)

2
12.56636
12.56636
10
314.159
62.8318
ValueError: Radius must be positive
AttributeError: can't set attribute
50.26544


# Worked On Descriptors Fields

In [40]:
class Field:
  def __init__(self,default=""):
    self.default=default

  def __set_name__ (self ,instance,name):
    self.name=name
    #print(self.name)

  def __set__(self,instance,value):
    #value=value.capitalize()
    if instance is None:
      #print("i am working")
      return self # since this is a function with this return statetement the below code will be neglected
    #if getattr(instance,"frozen",False):
     # raise AttributeError(f"{instance.__class__.__name__} is immutable")

    value=self._validate(value)
    """if self.name in instance.__dict__:
      #raise NotImplementedError(f"You cannot  changed  {self.name} of {instance} and has value of { instance.__dict__[self.name] } that is already created")
      raise NotImplementedError(f"Object created is not mutable")"""

    instance.__dict__[self.name] = value

  def __get__(self,instance,owner):
    if instance is None:
      return self
    return instance.__dict__.get(self.name,self.default)


In [41]:
# @title
# attribute assignment on instances using dot notation (instance.attribute = value).
class Textfield(Field):
  def __init__(self, default=""):
    super().__init__(default)

  def _validate (self,value):
    if not isinstance(value,str):
      raise TypeError(f"{value} is not A text")
    if value.isspace() or value =="" :
      raise ValueError(f"u must give a value ")
    return value.capitalize()

class ColorField(Textfield):
  def __init__(self, default="White"):
     super().__init__(default)
  def _validate (self,value):
    color_list = [
    "Red",
    "Orange",
    "Yellow",
    "Green",
    "Blue",
    "Indigo",
    "Violet",
    "Black",

    ]
    if not isinstance(value,str):
      raise TypeError(f"{value} is not A text and what should be is Color")
    value=value.capitalize()

    if value not in color_list:
      raise TypeError(f"{value} is not a color we want to create" )
    return value

class WholeNumber(Field):
  def __init__(self,min_value=0,max_value=100000,default=0):
      super().__init__(default)
      self.min_value=min_value
      self.max_value=max_value

  def _validate (self,value):
    if not isinstance(value,int):
      raise TypeError(f"{value} You are to input numbers only")
    if not(self.min_value<=  value  <=self.max_value):
        raise ValueError(f" {self.name} out of Range")
    return value

class PositiveNumber(Field):
  def __init__(self,default=0):
    super().__init__(default)

  def _validate(self,value):
    if not isinstance(value,(int,float)):
      raise TypeError(f"{value} You are to input numbers only")
    if value<=0:
      raise ValueError(f"{self.name} can never be negative")
    return value

class TrueOrFalse(Field):
  def __init__(self,default=False):
    self.default=False

  def _validate(self,value):
    if not isinstance(value,bool):
      raise ValueError("it is should either true or False")
    return value


In [42]:
class Test:
    brand = Textfield()
    model = Textfield()
    name = Textfield()
    age = WholeNumber(min_value=0, max_value=150)
    price = PositiveNumber()
    active = TrueOrFalse()
t = Test()

#Parent class that has instance of descriptor as its class attributes

MRO = Method Resolution Order. It's the ordered list of classes our child class has inherited from

> all class always inherit for class named object
> so i will use for loop and reversed func to collect those attribute i used reversed to get those atrributes in order




In [43]:
class BaseClass:
  def __init__(self,*args,**kwargs):
    self._frozen=False
    field=self.collect()
    names=[name for name in field.keys()]
    for name,value in zip(names,args):
      setattr(self,name,value)

    for name,value in kwargs.items():
      if name not in names:
        raise ValueError(f"{name} is not a define atrribute of {self.__class__.__name__}")
      setattr(self,name,value)
    setattr(self,"_frozen",True)
  @classmethod
  def collect(cls):
    field_collector={}
    #for name,attr in cls.__dict__.items(): what if the cls we want to collect it attribute is a child class and
    # we want to collect out the attribute also for that child class we will use .__mro__ method resolution order
    for Klass in reversed(cls.__mro__):
      #print(Klass)
      for attr,value in Klass.__dict__.items():
        if isinstance(value,Field):
          field_collector[attr]=value
      #print(field_collector)
    return field_collector
  def __setattr__(self, name, value) :
    if getattr(self,'_frozen',False):
      raise AttributeError(f"{self.__class__.__name__} is immutable")
    super().__setattr__(name,value)
  """def __init_subclass__(cls):
    #print("i wnt to runn")
    cls.field_collector=cls.collect()
    #print(cls.field_collector
    names=[name for name in cls.field_collector.keys()]
    print(names)

    def __init__(self,*args):
      print(args)
      for name,value in zip(names,args):
        setattr(self,name,value)
    cls.__init__= __init__"""
  def __repr__(self):
    save=[]
    for name in self.collect().keys():
      value =getattr(self,name)
      if name == "_frozen":
        continue
      save.append(f"{name}={value}")

    return f"{self.__class__.__name__}({','.join(save)})"
    #print(f"This is a {self.__name__}")
  def __eq__(self,second):
    if not isinstance(second,type(self)):
      return NotImplemented
    field=self.collect()
    for name in field.keys():
      if getattr(self,name)!=getattr(second,name):
        return False
    return True

#Child classes

##car

In [50]:
class Car(BaseClass):
    brand = Textfield()
    model = Textfield()
    year = WholeNumber(min_value=1900, max_value=2100)
    price = PositiveNumber()
    color = ColorField()
    mileage = PositiveNumber()

In [49]:
c1 = Car("Toyota", "Corolla", 2020, 15000, "red", 45000)
c2 = Car("Toyota", "Corolla", 2020, 15000, "red", 45000)
c3 = Car("Honda", "Civic", 2018, 12000, "blue", 60000)

print(repr(c1))
print(c1 == c2)              # True
print(c1 == c3)              # False
print(c1.brand, c1.color)    # Toyota Red

try:
    Car("", "Corolla", 2020, 15000, "red", 45000)
except ValueError as e:
    print("empty brand:", e)

try:
    Car("Toyota", "Corolla", 1800, 15000, "red", 45000)
except ValueError as e:
    print("bad year:", e)

try:
    Car("Toyota", "Corolla", 2020, -100, "red", 45000)
except ValueError as e:
    print("negative price:", e)

try:
    Car("Toyota", "Corolla", 2020, 15000, "pink", 45000)
except TypeError as e:
    print("bad color:", e)

try:
    c1.brand = "Honda"
except AttributeError as e:
    print("immutable:", e)

Car(brand=Toyota,model=Corolla,year=2020,price=15000,color=Red,mileage=45000)
True
False
Toyota Red
empty brand: u must give a value 
bad year:  year out of Range
negative price: price can never be negative
bad color: Pink is not a color we want to create
immutable: Car is immutable


##Point

In [ ]:
class Point(BaseClass):
    x = PositiveNumber()
    y = PositiveNumber()

In [ ]:
p1 = Point(3, 4)
p2 = Point(3, 4)
p3 = Point(3, 5)

print(repr(p1))              # Point(x=3,y=4)
print(p1.x, p1.y)            # 3 4
print(p1 == p2)              # True
print(p1 == p3)              # False
print(p1 == "not a point")   # False

try:
    p1.x = 10
except AttributeError as e:
    print("immutable:", e)

try:
    Point(-1, 4)
except ValueError as e:
    print("value:", e)

try:
    Point("hello", 4)
except TypeError as e:
    print("type:", e)

##Motorcycle

In [ ]:
class Motorcycle(BaseClass):
    brand = Textfield()
    model = Textfield()
    year = WholeNumber(min_value=1900, max_value=2100)
    price = PositiveNumber()
    color = ColorField()
    has_sidecar = TrueOrFalse()

In [ ]:
m1 = Motorcycle("Harley", "Sportster", 2019, 8000, "black", False)
m2 = Motorcycle("Harley", "Sportster", 2019, 8000, "black", False)
m3 = Motorcycle("Harley", "Sportster", 2019, 8000, "black", True)

print(repr(m1))
print(m1 == m2)              # True
print(m1 == m3)              # False
print(m1.has_sidecar)        # False

try:
    Motorcycle("Harley", "Sportster", 2019, 8000, "black", "yes")
except ValueError as e:
    print("bad bool:", e)

##Book

In [47]:
class Book(BaseClass):
    title = Textfield()
    author = Textfield()
    year_published = WholeNumber(min_value=1000, max_value=2100)
    isbn = Textfield()
    available = TrueOrFalse()

In [48]:
b1 = Book("Dune", "Frank Herbert", 1965, "9780441013593", True)
b2 = Book("Dune", "Frank Herbert", 1965, "9780441013593", True)
b3 = Book("Neuromancer", "William Gibson", 1984, "9780441569595", False)

print(repr(b1))
print(b1 == b2)              # True
print(b1 == b3)              # False
print(b1.title, b1.available)  # Dune True

try:
    Book("Dune", "Frank Herbert", 500, "9780441013593", True)
except ValueError as e:
    print("year too old:", e)

try:
    b1.available = False
except AttributeError as e:
    print("immutable:", e)

Book(title=Dune,author=Frank herbert,year_published=1965,isbn=9780441013593,available=True)
True
False
Dune True
year too old:  year_published out of Range
immutable: Book is immutable
